In [1]:
import datetime, time, random
import pandas as pd

In [2]:
def chop_microseconds(delta):
    return delta - datetime.timedelta(microseconds=delta.microseconds)


def run_ipl_simulation(basePoints_master, matches_master,
                       top_n=4, print_every=500000,
                       track_scenarios=True):
    """
    Same as v2, plus an optional tracking layer that records, for every
    (team, metric) pair, the scenario indices that contributed.

    A scenario is fully described by its index i in [0, 2^n). Bit k of i
    decides the winner of the k-th remaining match: bit = 0 means team1
    of the match string wins, bit = 1 means team2 wins. So given an
    index, we can rebuild the entire scenario without storing it.
    """
    basePoints = basePoints_master.copy()
    matches    = matches_master.copy()

    teams     = sorted(basePoints.keys())
    n         = len(matches)
    combos    = pow(2, n)
    formatStr = "0{}b".format(n)

    print("Pending matches    : {}".format(n))
    print("Possible scenarios : {:,}\n".format(combos))

    confirmed  = {t: 0   for t in teams}
    good_nrr   = {t: 0.0 for t in teams}
    confirmed2 = {t: 0   for t in teams}
    good_nrr2  = {t: 0.0 for t in teams}

    # Per-(team, metric) lists of scenario indices. Only populated if
    # track_scenarios is True. For 'good_nrr' metrics we record every
    # scenario where the team gets non-zero credit (ahead < cutoff);
    # the credit amount itself can always be recomputed from i.
    if track_scenarios:
        sc_top_n_confirmed = {t: [] for t in teams}
        sc_top_n_good_nrr  = {t: [] for t in teams}
        sc_top_2_confirmed = {t: [] for t in teams}
        sc_top_2_good_nrr  = {t: [] for t in teams}

    start_time = time.time()

    for i in range(combos):
        if print_every and (i + 1) % print_every == 0:
            elapsed = chop_microseconds(datetime.timedelta(seconds=time.time() - start_time))
            print("  {:,} / {:,} ({:.1f}%) | elapsed {}".format(
                i+1, combos, (i+1)*100/combos, elapsed))

        binaryStr = format(i, formatStr)
        simPoints = basePoints.copy()
        for matchCnt, seq in enumerate(binaryStr):
            team1, team2 = matches[matchCnt].split(":")
            simPoints[team1 if seq == "0" else team2] += 2

        for team in teams:
            my_pts = simPoints[team]
            ahead  = sum(1 for t in teams if t != team and simPoints[t] > my_pts)
            tied   = sum(1 for t in teams if t != team and simPoints[t] == my_pts)

            if ahead + tied <= top_n - 1:
                confirmed[team] += 1
                if track_scenarios:
                    sc_top_n_confirmed[team].append(i)

            if ahead < top_n:
                good_nrr[team] += min(1.0, (top_n - ahead) / (1 + tied))
                if track_scenarios:
                    sc_top_n_good_nrr[team].append(i)

            if ahead + tied <= 1:
                confirmed2[team] += 1
                if track_scenarios:
                    sc_top_2_confirmed[team].append(i)

            if ahead < 2:
                good_nrr2[team] += min(1.0, (2 - ahead) / (1 + tied))
                if track_scenarios:
                    sc_top_2_good_nrr[team].append(i)

    elapsed_total = chop_microseconds(datetime.timedelta(seconds=time.time() - start_time))

    summary = []
    for team in teams:
        summary.append({
            "team"               : team.upper(),
            "top2_confirmed_pct" : round(confirmed2[team] * 100.0 / combos, 6),
            "top2_good_nrr_pct"  : round(good_nrr2[team]  * 100.0 / combos, 6),
            "top4_confirmed_pct" : round(confirmed[team]  * 100.0 / combos, 6),
            "top4_good_nrr_pct"  : round(good_nrr[team]   * 100.0 / combos, 6),
        })
    # summary.sort(key=lambda x: -x["top2_good_nrr_pct"])
    summary.sort(key=lambda x: (x["top4_confirmed_pct"], x["top4_good_nrr_pct"]), reverse=True)

    out = {
        "summary"         : summary,
        "total_scenarios" : combos,
        "elapsed"         : str(elapsed_total),
        "matches"         : list(matches_master),
        "basePoints"      : dict(basePoints_master),
        "top_n"           : top_n,
    }
    if track_scenarios:
        # Use upper-case team keys for consistency with the summary table.
        out["scenarios"] = {
            "top{}_confirmed".format(top_n) : {t.upper(): sc_top_n_confirmed[t] for t in teams},
            "top{}_good_nrr".format(top_n)  : {t.upper(): sc_top_n_good_nrr[t]  for t in teams},
            "top2_confirmed"                 : {t.upper(): sc_top_2_confirmed[t] for t in teams},
            "top2_good_nrr"                  : {t.upper(): sc_top_2_good_nrr[t]  for t in teams},
        }
    print("Completed {:,} scenarios in {}\n".format(combos, elapsed_total))
    return out


def decode_scenario(i, basePoints, matches):
    """
    Given a scenario index i, return everything you might want to know
    about that scenario:
      - 'winners'  : a flat list of winner team names, one entry per
                     remaining match, in the same order as `matches`.
                     e.g. ["kkr", "gt",......  ......, "dc"].
      - 'outcomes' : list of dicts {match, winner, loser}, in case you
                     want both teams of each match handy.
      - 'points'   : final simulated points table after applying every
                     match outcome to the starting basePoints.
      - 'standings': list of (team, points) sorted by points desc.

    All four are derived from the same single bit-walk over `i`, so
    decoding a scenario is O(n) where n is the number of pending matches.
    """
    n         = len(matches)
    binaryStr = format(i, "0{}b".format(n))
    simPoints = dict(basePoints)
    winners   = []
    outcomes  = []
    for k, seq in enumerate(binaryStr):
        team1, team2 = matches[k].split(":")
        # Bit convention: 0 means team1 wins, 1 means team2 wins. This
        # has to match the convention used in run_ipl_simulation, which
        # it does — same `if seq == "0"` branch.
        if seq == "0":
            winner, loser = team1, team2
        else:
            winner, loser = team2, team1
        simPoints[winner] += 2
        winners.append(winner)
        outcomes.append({"match": matches[k], "winner": winner, "loser": loser})
    standings = sorted(simPoints.items(), key=lambda kv: -kv[1])
    return {"index": i, "winners": winners, "outcomes": outcomes,
            "points": simPoints, "standings": standings}


def view_scenarios(results, team, metric, limit=10, show_winners=True):
    """
    Return a pandas DataFrame summarising the qualifying scenarios for
    (team, metric). Each row is one scenario.

    The `limit` parameter controls how many scenarios are returned:
      - limit = N (a positive integer) : return the first N scenarios.
      - limit = None                   : return ALL qualifying scenarios.
                                         Useful when you want to export
                                         or analyse the full set.
      - limit = 0                      : return an empty DataFrame
                                         (just the schema).

    Note on performance: for popular metrics like SRH's top4_good_nrr,
    `limit=None` may need to decode hundreds of thousands of scenarios.
    Each decode is fast (~20 string ops), so a few hundred thousand rows
    take a few seconds and a few hundred MB of RAM. If you hit memory
    pressure, write directly to disk in chunks instead of building one
    huge DataFrame in memory.

    Columns:
      - 'i'                 : the scenario index
      - '{team}_credit'     : 1.0 for confirmed metrics, possibly
                              fractional (e.g. 0.25) for good_nrr
      - '{team}_rank_min'   : best-case finishing rank in this scenario
      - '{team}_rank_max'   : worst-case finishing rank
      - one column per team : final simulated points
      - 'winners'           : (if show_winners) list of winners aligned
                              with results['matches'], so element k is
                              the winner of the k-th remaining match.

    Returns:
      (DataFrame, total_count) where total_count is the FULL number of
      qualifying scenarios — not the row count of the returned frame.
      That way, even if you ask for limit=10, you can see how many there
      are in total.
    """
    indices    = results["scenarios"][metric][team.upper()]
    matches    = results["matches"]
    basePoints = results["basePoints"]
    cutoff     = results["top_n"] if metric.startswith("top4") else 2

    # Decide how many scenarios to actually decode.
    # Python's slicing semantics let us treat None and a positive int
    # uniformly: indices[:None] is the full list, indices[:5] is the
    # first 5. So we can just pass `limit` directly into the slice.
    # The only special case worth handling is the user passing 0,
    # which Python would handle correctly anyway but we make explicit
    # for clarity below.
    selected = indices[:limit] if limit != 0 else []

    # Friendly heads-up when the export is large, so the user doesn't
    # think the cell is hung. We only print once per call so it stays
    # quiet for normal use.
    if limit is None and len(selected) > 100_000:
        print(f"Decoding all {len(selected):,} scenarios for "
              f"{team.upper()} / {metric}; this may take a few seconds.")

    rows = []
    for i in selected:
        d      = decode_scenario(i, basePoints, matches)
        my_pts = d["points"][team.lower()]
        ahead  = sum(1 for t, p in d["points"].items() if t != team.lower() and p > my_pts)
        tied   = sum(1 for t, p in d["points"].items() if t != team.lower() and p == my_pts)
        if "confirmed" in metric:
            credit = 1.0
        else:
            credit = min(1.0, (cutoff - ahead) / (1 + tied)) if ahead < cutoff else 0.0

        row = {"i": i,
               f"{team.upper()}_credit"  : credit,
               f"{team.upper()}_rank_min": ahead + 1,
               f"{team.upper()}_rank_max": ahead + 1 + tied}
        for t, p in sorted(d["points"].items()):
            row[t.upper()] = p
        if show_winners:
            row["winners"] = d["winners"]
        rows.append(row)
    return pd.DataFrame(rows), len(indices)

In [3]:
# ---------- run and verify ----------
basePoints = {
    "rcb": 18, 
    "gt": 16,
    "srh": 14, 
    "pbks": 13, 
    "rr": 12, 
    "csk": 12, 
    "dc": 12, 
    "kkr": 11, 
    "mi": 8, 
    "lsg": 8,
}
matches = [
    "csk:srh", 
    "rr:lsg", 
    "kkr:mi",
    "gt:csk", 
    "srh:rcb", 
    "lsg:pbks", 
    "mi:rr", 
    "kkr:dc",
]
results = run_ipl_simulation(basePoints, matches, top_n=4)

Pending matches    : 8
Possible scenarios : 256

Completed 256 scenarios in 0:00:00



In [4]:
summary = pd.DataFrame(results["summary"])
summary

,team,top2_confirmed_pct,top2_good_nrr_pct,top4_confirmed_pct,top4_good_nrr_pct
0,RCB,87.5,95.833333,100.000000,100.000000
1,GT,37.5,61.197917,96.875000,99.218750
2,SRH,12.5,30.468750,72.656250,80.143229
3,PBKS,0.0,0.000000,26.953125,31.054688
4,CSK,0.0,9.635417,23.437500,33.528646
5,RR,0.0,2.864583,23.437500,33.203125
6,KKR,0.0,0.000000,9.765625,13.867188
7,DC,0.0,0.000000,1.562500,8.984375
8,LSG,0.0,0.000000,0.000000,0.000000
9,MI,0.0,0.000000,0.000000,0.000000


In [5]:
# top2_good_nrr, top2_confirmed, top4_good_nrr, top4_confirmed
df, total = view_scenarios(results, "DC", "top4_confirmed", limit=None)
print(df.shape)
total

(4, 15)


4

In [6]:
df

,i,DC_credit,DC_rank_min,DC_rank_max,CSK,DC,GT,KKR,LSG,MI,PBKS,RCB,RR,SRH,winners
0,193,1.0,4,4,12,14,18,13,12,10,13,18,12,18,"[srh, lsg, kkr, gt, srh, lsg, mi, dc]"
1,201,1.0,4,4,12,14,18,13,12,10,13,20,12,16,"[srh, lsg, kkr, gt, rcb, lsg, mi, dc]"
2,225,1.0,4,4,12,14,18,11,12,12,13,18,12,18,"[srh, lsg, mi, gt, srh, lsg, mi, dc]"
3,233,1.0,4,4,12,14,18,11,12,12,13,20,12,16,"[srh, lsg, mi, gt, rcb, lsg, mi, dc]"


In [7]:
dc_conf = results["scenarios"]["top4_confirmed"]["DC"]
print("List of scenario indices for DC's top4_confirmed:\n", dc_conf)

random_choice = random.choice(dc_conf)
print("\nDecoding a random scenario for DC's top4_confirmed:\n", random_choice)
d = decode_scenario(random_choice, basePoints, matches)

List of scenario indices for DC's top4_confirmed:
 [193, 201, 225, 233]

Decoding a random scenario for DC's top4_confirmed:
 201


In [8]:
print(f"\nPaired with the matches:")
for match, matchwinner in zip(matches, d["winners"]):
    loser = [t for t in match.split(":") if t != matchwinner][0]
    print(f"  {match.upper():>10}  ->  {matchwinner.upper():>4}  beats  {loser.upper():>4}")

print(f"\nFinal standings:")
for t, p in d["standings"]:
    print(f"  {t.upper():>4}: {p}")


Paired with the matches:
     CSK:SRH  ->   SRH  beats   CSK
      RR:LSG  ->   LSG  beats    RR
      KKR:MI  ->   KKR  beats    MI
      GT:CSK  ->    GT  beats   CSK
     SRH:RCB  ->   RCB  beats   SRH
    LSG:PBKS  ->   LSG  beats  PBKS
       MI:RR  ->    MI  beats    RR
      KKR:DC  ->    DC  beats   KKR

Final standings:
   RCB: 20
    GT: 18
   SRH: 16
    DC: 14
  PBKS: 13
   KKR: 13
    RR: 12
   CSK: 12
   LSG: 12
    MI: 10


In [9]:
def run_bottom_simulation(results, bottom_m=1, print_every=500000, track_scenarios=True):
    """
    Mirror of run_ipl_simulation for the bottom of the table.
    Reuses the scenario indexing from the existing results (same basePoints/matches),
    so scenario indices i are directly compatible with decode_scenario and view_scenarios.

    Conditions (mirroring top_n_confirmed / top_n_good_nrr):
      - bottom_m_confirmed : behind + tied <= bottom_m - 1
            (for m=1: alone at the lowest point total -> last for sure)
      - bottom_m_bad_nrr   : behind < bottom_m, credit = min(1, (bottom_m - behind)/(1 + tied))
            (for m=1: nobody strictly below me; credit shared equally among bottom-tied teams)
    """
    basePoints = dict(results["basePoints"])
    matches    = list(results["matches"])
    teams      = sorted(basePoints.keys())
    n          = len(matches)
    combos     = pow(2, n)
    formatStr  = "0{}b".format(n)

    print("Pending matches    : {}".format(n))
    print("Possible scenarios : {:,}\n".format(combos))

    confirmed_bot = {t: 0   for t in teams}
    bad_nrr_bot   = {t: 0.0 for t in teams}

    if track_scenarios:
        sc_bot_confirmed = {t: [] for t in teams}
        sc_bot_bad_nrr   = {t: [] for t in teams}

    start_time = time.time()

    for i in range(combos):
        if print_every and (i + 1) % print_every == 0:
            elapsed = datetime.timedelta(seconds=time.time() - start_time)
            elapsed = elapsed - datetime.timedelta(microseconds=elapsed.microseconds)
            print("  {:,} / {:,} ({:.1f}%) | elapsed {}".format(
                i+1, combos, (i+1)*100/combos, elapsed))

        binaryStr = format(i, formatStr)
        simPoints = basePoints.copy()
        for matchCnt, seq in enumerate(binaryStr):
            team1, team2 = matches[matchCnt].split(":")
            simPoints[team1 if seq == "0" else team2] += 2

        for team in teams:
            my_pts = simPoints[team]
            behind = sum(1 for t in teams if t != team and simPoints[t] <  my_pts)
            tied   = sum(1 for t in teams if t != team and simPoints[t] == my_pts)

            # bottom_m_confirmed: alone or tied with at most (bottom_m-1) others at the lowest pts
            if behind + tied <= bottom_m - 1:
                confirmed_bot[team] += 1
                if track_scenarios:
                    sc_bot_confirmed[team].append(i)

            # bottom_m_bad_nrr: nobody (strictly enough) below me; fractional credit
            if behind < bottom_m:
                bad_nrr_bot[team] += min(1.0, (bottom_m - behind) / (1 + tied))
                if track_scenarios:
                    sc_bot_bad_nrr[team].append(i)

    elapsed_total = datetime.timedelta(seconds=time.time() - start_time)
    elapsed_total = elapsed_total - datetime.timedelta(microseconds=elapsed_total.microseconds)

    summary = []
    for team in teams:
        summary.append({
            "team"                   : team.upper(),
            "bottom_one_pct"         : round(confirmed_bot[team] * 100.0 / combos, 6),
            "bottom_one_bad_nrr_pct" : round(bad_nrr_bot[team]   * 100.0 / combos, 6),
        })
    # Sort so the team most at risk of finishing last is on top.
    summary.sort(key=lambda x: -x["bottom_one_bad_nrr_pct"])

    out = {
        "summary"         : summary,
        "total_scenarios" : combos,
        "elapsed"         : str(elapsed_total),
        "matches"         : matches,
        "basePoints"      : basePoints,
        "bottom_m"        : bottom_m,
    }
    if track_scenarios:
        out["scenarios"] = {
            "bottom{}_confirmed".format(bottom_m) : {t.upper(): sc_bot_confirmed[t] for t in teams},
            "bottom{}_bad_nrr".format(bottom_m)   : {t.upper(): sc_bot_bad_nrr[t]   for t in teams},
        }
    print("Completed {:,} scenarios in {}\n".format(combos, elapsed_total))
    return out

In [10]:
bottom_results = run_bottom_simulation(results, bottom_m=1)
bottom_df = pd.DataFrame(bottom_results["summary"])
bottom_df['team'] = pd.Categorical(bottom_df['team'], categories=summary['team'], ordered=True)
bottom_df = bottom_df.sort_values('team')
bottom_df = bottom_df.reset_index(drop=True)
bottom_df.rename(columns={"bottom_one_pct": "Bottom confirmed %", "bottom_one_bad_nrr_pct": "Bottom with bad NRR %"}, inplace=True)
bottom_df

Pending matches    : 8
Possible scenarios : 256

Completed 256 scenarios in 0:00:00



,team,Bottom confirmed %,Bottom with bad NRR %
0,RCB,0.000,0.000000
1,GT,0.000,0.000000
2,SRH,0.000,0.000000
3,PBKS,0.000,0.000000
4,CSK,0.000,0.156250
5,RR,0.000,0.742188
6,KKR,3.125,3.125000
7,DC,0.000,0.742188
8,LSG,31.250,47.617188
9,MI,31.250,47.617188


In [11]:
bottom_df.to_csv("bottom_scenarios_base.csv", index=False)